*Notebook 03 of 12 in `notebooks/scripts_text/`. Parameters: `analysis/scripts_text/config.yaml`; decisions and run notes: `analysis/scripts_text/LOG.md`. The static page `site/scripts/index.html` is built from the same outputs.*

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # notebooks/scripts_text -> repo root
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from analysis.scripts_text import s03_clean as step

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Step 3 · Cleaning what isn't language

**What we do.** We remove what a reader wouldn't count as the script's language, and count every removal:

- **split contractions** joined back ("do n't" → "don't"), because some scripts in MovieSum are pre-tokenised;
- **scene headings** (INT., EXT., DAY, NIGHT) and **technical directions** (CUT TO, FADE IN, V.O., O.S., CONT'D);
- **page and revision artefacts** (page numbers, "Revised pages", "Written by …");
- **character names**, taken from each script's own speaker cues.

**Why names.** A character's name is unique to one film, so in a two-group comparison it is always
"distinctive" and tells us nothing about success. In the course's Marvel text, names dominated every
word list for this reason.

**Extraction is not resolution.** The speaker cues are raw strings ("JULES (V.O.)", "COP #2", "OLD MAN"). We
strip extensions and titles and split them into tokens, but we don't resolve nicknames ("Mike" vs "Michael").
Some name tokens are ordinary words (Will, Grace, Hope, Frank): those are removed only where they are
capitalised in the middle of a sentence. As a second source of names we run spaCy's named-entity recogniser
(NER) on the dialogue and report how much it would add, without using it in the primary analysis.

**Preprocessing is part of the model.** Every choice below changes which words can appear distinctive later.
We keep stopwords (function words like "the", "you", "we" carry style) and don't lemmatise (lemmas merge
"killed" and "kills", which may be the signal), and run both the other way as variants.

**What it measures.** The vocabulary that is left: tokens (every occurrence), types (distinct words) and hapax
types (words occurring once). Tokens ≥ types ≥ hapax always.

**What it misses.** Our tokenizer splits on anything that isn't a letter, so "ice-cream" becomes two words
and numbers disappear. The name list misses characters who never speak and keeps nicknames unresolved.

In [ ]:
res = step.run()   # recomputes this step and saves its outputs

**Table.** Every preprocessing choice, its setting and why.

In [ ]:
display(step.tables(res)["choices"])

,choice,setting,why
0,tokenizer,[A-Za-z]+(?:'[A-Za-z]+)?,"same as the reviews analysis; keeps don't, it's as one word"
1,lowercase,yes,'The' and 'the' are one word for our question
2,split contractions,joined ('do n't' → 'don't'),some MovieSum scripts are pre-tokenised; without this 'n't' becomes a word
3,scene headings,removed (full layer),"INT., EXT., DAY, NIGHT describe format, not language"
4,technical directions,"removed (CUT TO, FADE IN, V.O., O.S., CONT'D …)","formatting, and some writers use far more of it"
5,page and revision artefacts,"lines removed (page numbers, 'Revised …', 'Written by …')",not part of the story
6,character names,removed (from each script's own speaker cues),names are unique to a film and would dominate the comparison
7,names that are words,removed only when capitalised mid-sentence,keeps 'will' the verb while removing 'Will' the character
8,minimum token length,2 letters,single letters are mostly initials and artefacts
9,stopwords,"kept (primary); removed (variant, spaCy list)",function words carry style; removing them is a modelling choice


**Table.** Tokens, types and hapax types of all dialogue after each stage. Stage 2 is the primary corpus; 2b, 3a and 3b are variants.

In [ ]:
display(step.tables(res)["stages"])

,stage,tokens,types,hapax,min_length_rule
0,0 raw dialogue (Step 2),10036412,109566,46609,tokens shorter than 2 letters dropped from stage 2 on
1,"1 contractions joined, technical directions and artefact lines removed",9969219,109585,46621,tokens shorter than 2 letters dropped from stage 2 on
2,2 character names removed (labels),9186469,104055,45818,tokens shorter than 2 letters dropped from stage 2 on
3,"2b names removed (labels + NER), check only",9030034,92076,40692,tokens shorter than 2 letters dropped from stage 2 on
4,3a + stopwords removed (variant),4322216,103754,45817,tokens shorter than 2 letters dropped from stage 2 on
5,3b + lemmatised (variant),8999645,87442,40646,tokens shorter than 2 letters dropped from stage 2 on


**Figure.** The same counts as bars on a log axis. Names remove many types but few tokens; stopwords remove few types but many tokens.

In [ ]:
step.figures(res)["stages"].show()

**Table.** The 30 most frequent dialogue words before cleaning, after cleaning (primary) and with stopwords removed. Before cleaning, names and formatting can appear; after, the list is ordinary spoken English.

In [ ]:
display(step.tables(res)["top"])

,rank,before (raw),before count,after (primary),after count,"after, stopwords removed"
0,1,you,349520,you,348477,i'm
1,2,the,344240,the,344190,don't
2,3,i,283584,to,242343,know
3,4,to,243175,and,157950,it's
4,5,a,219285,it,131424,like
5,6,and,157954,of,128873,you're
6,7,it,131784,in,108604,got
7,8,of,128875,that,101190,that's
8,9,in,109135,is,98716,right
9,10,is,101671,me,87973,want


**Figure.** Rank against frequency on log-log axes for the raw, primary and stopwords-removed dialogue. A roughly straight line is suggestive of a Zipf-like heavy tail: a few words are extremely common and most are rare. It is not proof of Zipf's law; testing that would need a fitted model against alternatives.

In [ ]:
step.figures(res)["rankfreq"].show()

**Private check.** 20 scripts at random (fixed seed) with the name tokens found in their speaker cues and the extra ones NER suggests. Check: are these names, and is anything missing? Notebook only, output cleared before commit.

In [ ]:
display(step.private(res)["names_sample"])   # output cleared before commit

**Plain-language reading** (generated from the saved numbers):

In [ ]:
display(Markdown(step.reading(res)))

Cleaning joined 339,595 split contractions; 483 scripts are pre-tokenised (20 or more split contractions joined). It removed 167,248 tokens of technical directions and 10,146 tokens on page or revision lines across all layers, and 713,381 heading tokens from the full layer. Speaker cues give a median of 55 name tokens per script; 48,294 of the 74,915 are also ordinary English words, which is why those are removed only when capitalised mid-sentence. In the dialogue, 233,914 name tokens go (2.3% of all dialogue tokens). spaCy's NER finds a median of 74 person-name tokens per script, 58 of them not among the speaker cues (people who are talked about but never speak, and NER errors); adding them would remove 156,435 more tokens. The primary dialogue corpus has 9,186,469 tokens, 104,055 types and 45,818 hapax types (tokens ≥ types ≥ hapax holds at every stage).